# EDA: Transfers and Ratings
This notebook profiles the `transfers` and `ratings` datasets: datatypes, missing values, distributions, correlations, and some feature engineering for `position` and `league` when relevant.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
from pathlib import Path
sns.set(style="whitegrid")
pd.options.display.max_columns = 200
pd.options.display.width = 160

In [ ]:
# Locate zip files in data/ and list their contents
data_dir = Path('..') / 'data'  # notebook is in notebooks/ so ../data
for p in data_dir.glob('*.zip'):
    print('ZIP:', p.name)
    with zipfile.ZipFile(p) as z:
        print('  contains ->', z.namelist())

In [ ]:
# Adjust filenames below if the zip contains different internal names
transfers_zip = data_dir / 'transfers.zip'
ratings_zip = data_dir / 'ratings.zip'
# Pick first file inside each zip by default
with zipfile.ZipFile(transfers_zip) as z:
    transfers_name = z.namelist()[0]
with zipfile.ZipFile(ratings_zip) as z:
    ratings_name = z.namelist()[0]
print('using', transfers_name, 'and', ratings_name)
# Read specific files from the zip archives
with zipfile.ZipFile(transfers_zip) as z:
    with z.open(transfers_name) as f:
        transfers = pd.read_csv(f)
with zipfile.ZipFile(ratings_zip) as z:
    # ratings zip may contain multiple season files; we start with the first
    with z.open(ratings_name) as f:
        ratings = pd.read_csv(f)
print('transfers', transfers.shape)
print('ratings', ratings.shape)

## Helper: column profiling function
This helper prints datatype, missingness and overview statistics, and creates distribution plots appropriate to the column type.

In [ ]:
def profile_column(df, col, figsize=(6,3)):
    s = df[col]
    dtype = s.dtype
    missing = s.isna().sum()
    pct_missing = 100 * missing / len(df)
    nunique = s.nunique(dropna=True)
    print(f'Column: {col}')
    print(' - dtype:', dtype)
    print(f' - missing: {missing} ({pct_missing:.2f}%)')
    print(' - unique (non-null):', nunique)
    if pd.api.types.is_numeric_dtype(s):
        print(s.describe())
        fig, ax = plt.subplots(1,2, figsize=(12,3))
        sns.histplot(s.dropna(), ax=ax[0], kde=True)
        sns.boxplot(x=s.dropna(), ax=ax[1])
        plt.suptitle(col)
        plt.show()
    else:
        # Categorical / object
        top = s.value_counts(dropna=False).head(20)
        display(top)
        plt.figure(figsize=figsize)
        sns.barplot(x=top.values, y=top.index)
        plt.title(col)
        plt.show()

## Profile: transfers dataset (all features)
We'll iterate every column and show basic stats and distributions.

In [ ]:
# Quick overview
transfers.head()

In [ ]:
# Data types and missingness table for transfers
transfers_info = pd.DataFrame({
    'dtype': transfers.dtypes.astype(str),
    'n_missing': transfers.isna().sum(),
    'pct_missing': 100 * transfers.isna().mean(),
    'n_unique': transfers.nunique(dropna=True)
})
transfers_info.sort_values('pct_missing', ascending=False)

In [ ]:
# Profile each column (show first 6 to avoid too long output)
for i, col in enumerate(transfers.columns):
    if i >= 6:
        break
    profile_column(transfers, col)
    print('\n---\n')
# If you want to profile all columns uncomment below (may be long)
# for col in transfers.columns:
#     profile_column(transfers, col)
#     print('\n---\n')

### Numeric correlation matrix (transfers)
We compute Pearson correlations for numeric features and visualize a heatmap.

In [ ]:
num_cols = transfers.select_dtypes(include=[np.number]).columns.tolist()
corr = transfers[num_cols].corr()
plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Transfers: numeric correlations')
plt.show()

## Additional profiling: missingness map and duplicates (transfers)

In [ ]:
plt.figure(figsize=(12,3))
sns.heatmap(transfers.isna(), cbar=False)
plt.title('Missingness map: transfers')
plt.show()
print('Duplicate rows:', transfers.duplicated().sum())

## Profile: ratings dataset (selected features)
We focus on: `short_name`, `age`, `club`, `overall`, `potential`, `season`, `nationality`, `value_eur`, `position` (split to multiple), `league` (infer if possible).

In [ ]:
ratings.head()

In [ ]:
# Show info for selected features (if they exist)
sel = ['short_name','age','club','overall','potential','season','nationality','value_eur','position','league']
sel_existing = [c for c in sel if c in ratings.columns]
sel_existing, 'missing from ratings:' , [c for c in sel if c not in ratings.columns]

In [ ]:
# Data types & missingness for selected features
ratings_info = pd.DataFrame({
    'dtype': ratings.dtypes.astype(str),
    'n_missing': ratings.isna().sum(),
    'pct_missing': 100 * ratings.isna().mean(),
    'n_unique': ratings.nunique(dropna=True)
})
ratings_info.loc[sel_existing].sort_values('pct_missing', ascending=False)

### Profile numeric distributions for `age`, `overall`, `potential`, `value_eur` (when present)

In [ ]:
ratings['value_eur'].value_counts()

In [ ]:
for col in ['age','overall','potential','value_eur']:
    if col in ratings.columns:
        print('\n==', col, '==')
        profile_column(ratings, col)

### Position feature engineering
If `position` is a delimited string of multiple roles, we expand into indicator columns for the common positions.

In [ ]:
if 'position' in ratings.columns:
    # Assume positions are separated by '/' or ',' or ';' or '|'
    pos_series = ratings['position'].fillna('')
    split_pos = pos_series.str.replace(r'\s*,\s*', '/', regex=True)
    split_pos = split_pos.str.replace(r'\s*;\s*', '/', regex=True)
    split_pos = split_pos.str.replace(r'\s*\|\s*', '/', regex=True)
    positions = set()
    for s in split_pos.unique():
        if s:
            for p in str(s).split('/'):
                positions.add(p.strip())
    positions = sorted([p for p in positions if p])
    print('Detected positions (sample):', positions[:30])
    # create indicators for top positions
    top_positions = positions[:20]  # limit to first 20 unique tokens
    for p in top_positions:
        ratings[f'pos_{p}'] = split_pos.str.contains(rf'\b{re.escape(p)}\b', regex=True)
    print('Created indicator columns for top positions')
else:
    print('No position column found')

### League inference (attempt)
If `league` is not present, try to infer from `transfers` by mapping club -> league where possible.

In [ ]:
if 'league' not in ratings.columns:
    # Try mapping from transfers if transfers has a league-like column and club field
    possible_club_cols = [c for c in transfers.columns if 'club' in c.lower() or 'team' in c.lower()]
    possible_league_cols = [c for c in transfers.columns if 'league' in c.lower() or 'division' in c.lower()]
    print('transfers club-like cols:', possible_club_cols)
    print('transfers league-like cols:', possible_league_cols)
    if possible_club_cols and possible_league_cols:
        club_col = possible_club_cols[0]
        league_col = possible_league_cols[0]
        mapping = transfers[[club_col, league_col]].dropna().drop_duplicates().set_index(club_col)[league_col].to_dict()
        # apply mapping where club names match exactly
        ratings['inferred_league'] = ratings['club'].map(mapping)
        n_inferred = ratings['inferred_league'].notna().sum()
        print('Inferred league for', n_inferred, 'rows')
    else:
        print('Could not find club/league columns in transfers to infer league')
else:
    print('ratings already contains league column')

In [ ]:
# Inspect inferred_league results
if 'inferred_league' in ratings.columns:
    n_inferred = ratings['inferred_league'].notna().sum()
    print('Inferred leagues for', n_inferred, 'rows (of', len(ratings), ')')
    display(ratings['inferred_league'].value_counts().head(20))
    display(ratings[['club','inferred_league']].dropna().drop_duplicates().head(30))
else:
    print('No inferred_league column found')

### Pairwise correlations (ratings numeric features)
Compute correlations for numeric features such as `age`, `overall`, `potential`, `value_eur`.

In [ ]:
num_cols_r = [c for c in ['age','overall','potential','value_eur'] if c in ratings.columns]
if num_cols_r:
    corr_r = ratings[num_cols_r].corr()
    plt.figure(figsize=(6,4))
    sns.heatmap(corr_r, annot=True, cmap='coolwarm', center=0)
    plt.title('Ratings: numeric correlations')
    plt.show()
else:
    print('No target numeric columns present')

In [ ]:
# Summarize findings succinctly
print('Transfers columns summary:')
display(transfers_info.head(50))
print('\nRatings selected features summary:')
display(ratings_info.loc[sel_existing])

## Next steps / notes
- Run the full `profile_column` loop over all transfers columns (commented) if you want every column expanded.
- Consider normalizing `value_eur` (large skew) with log1p for modeling.
- For `position`, consider multi-hot encoding for machine learning.
- For `league`, obtaining a reliable external mapping (club -> league) improves analyses across seasons.

## Enhanced correlations — Spearman & log-scale

Pearson is sensitive to outliers and right-skewed distributions. Transfer fees and market values are heavily skewed, so we add:
- **Spearman** rank correlation — robust to skew and outliers
- **Pearson on log-transformed** values — removes scale distortion from large fees

Comparing all three gives a fuller picture of the true relationships.

In [ ]:
# Transfers: Pearson (raw), Spearman (rank), Pearson on log1p-transformed
num_cols_t = transfers.select_dtypes(include=[np.number]).columns.tolist()
transfers_num = transfers[num_cols_t].dropna()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.heatmap(transfers_num.corr(method='pearson'), annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=axes[0])
axes[0].set_title('Transfers: Pearson (raw)')

sns.heatmap(transfers_num.corr(method='spearman'), annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=axes[1])
axes[1].set_title('Transfers: Spearman (rank)')

transfers_log = np.log1p(transfers_num.clip(lower=0))
sns.heatmap(transfers_log.corr(method='pearson'), annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=axes[2])
axes[2].set_title('Transfers: Pearson (log1p)')

plt.tight_layout()
plt.show()

In [ ]:
# Ratings: Pearson (raw), Spearman (rank), Pearson on log1p-transformed
num_cols_r = [c for c in ['age', 'overall', 'potential', 'value_eur'] if c in ratings.columns]
ratings_num = ratings[num_cols_r].dropna()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.heatmap(ratings_num.corr(method='pearson'), annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=axes[0])
axes[0].set_title('Ratings: Pearson (raw)')

sns.heatmap(ratings_num.corr(method='spearman'), annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=axes[1])
axes[1].set_title('Ratings: Spearman (rank)')

ratings_log = np.log1p(ratings_num.clip(lower=0))
sns.heatmap(ratings_log.corr(method='pearson'), annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=axes[2])
axes[2].set_title('Ratings: Pearson (log1p)')

plt.tight_layout()
plt.show()

In [ ]:
# Pairplots — show scatter distributions between all numeric feature pairs

# Transfers
transfers_pair = transfers[['Age', 'Market_value', 'Transfer_fee']].dropna()
g = sns.pairplot(transfers_pair, diag_kind='kde', plot_kws={'alpha': 0.3})
g.fig.suptitle('Transfers: pairplot of numeric features', y=1.02)
plt.show()

# Ratings (sample for speed)
ratings_pair = (ratings[['age', 'overall', 'potential', 'value_eur']]
                .dropna()
                .sample(min(3000, len(ratings)), random_state=42))
g2 = sns.pairplot(ratings_pair, diag_kind='kde', plot_kws={'alpha': 0.15})
g2.fig.suptitle('Ratings: pairplot of numeric features (3 000-row sample)', y=1.02)
plt.show()

## Cross-dataset correlations (merged transfers × FIFA ratings)

Join transfers (seasons 2015–2018) with FIFA ratings on **last name + season + club** to produce a merged dataset. This lets us examine how FIFA ability ratings (`overall`, `potential`) correlate with actual `Transfer_fee` alongside `Age` and `Market_value`.

> **Merge note:** We use a simple last-name + season + club match (no fuzzy matching), which is fast and reproducible. Expect ~400–700 matched rows — sufficient for correlation analysis.

In [ ]:
# ── Load FIFA seasons 2015-2018 ──────────────────────────────────────────────
season_files = {
    'players_15.csv': 2015, 'players_16.csv': 2016,
    'players_17.csv': 2017, 'players_18.csv': 2018,
}

frames = []
with zipfile.ZipFile(ratings_zip) as z:
    available = z.namelist()
    for fname, yr in season_files.items():
        if fname in available:
            with z.open(fname) as f:
                df = pd.read_csv(f, low_memory=False)
            nat_col = next(
                (c for c in ['nationality_name', 'nationality'] if c in df.columns), None
            )
            keep = ['short_name', 'club', 'overall', 'potential']
            if nat_col:
                keep.append(nat_col)
            df = df[keep].copy()
            df['Season_yr'] = yr
            if nat_col and nat_col != 'Nationality':
                df = df.rename(columns={nat_col: 'Nationality'})
            frames.append(df)

fifas = pd.concat(frames, ignore_index=True)
fifas['Lastname'] = fifas['short_name'].str.split().str[-1]

for col in ['Lastname', 'club']:
    fifas[col] = (fifas[col]
                  .str.normalize('NFKD')
                  .str.encode('ascii', errors='ignore')
                  .str.decode('utf-8'))

# ── Prepare transfers 2015-2018 ───────────────────────────────────────────────
transfers_t = transfers.copy()
transfers_t['Season_yr'] = transfers_t['Season'].str[:4].astype(int, errors='ignore')
transfers_t = transfers_t[transfers_t['Season_yr'].between(2015, 2018)].copy()
transfers_t['Lastname'] = transfers_t['Name'].str.split().str[-1]
transfers_t['Transfer_fee_mln'] = transfers_t['Transfer_fee'] / 1e6
transfers_t['Market_value_mln'] = transfers_t['Market_value'] / 1e6

for col in ['Lastname', 'Team_from', 'Team_to']:
    transfers_t[col] = (transfers_t[col]
                        .str.normalize('NFKD')
                        .str.encode('ascii', errors='ignore')
                        .str.decode('utf-8'))

# ── Merge on Lastname + Season_yr + Team_from (primary), Team_to (fallback) ──
merged_primary = transfers_t.merge(
    fifas, how='inner',
    left_on=['Lastname', 'Season_yr', 'Team_from'],
    right_on=['Lastname', 'Season_yr', 'club'],
)
matched_idx = set(merged_primary['Name'].tolist())

unmatched = transfers_t[~transfers_t['Name'].isin(matched_idx)].copy()
merged_fallback = unmatched.merge(
    fifas, how='inner',
    left_on=['Lastname', 'Season_yr', 'Team_to'],
    right_on=['Lastname', 'Season_yr', 'club'],
)

merged_full = (pd.concat([merged_primary, merged_fallback], ignore_index=True)
               .dropna(subset=['overall', 'potential', 'Transfer_fee_mln']))

print(f'Matched rows: {len(merged_full)}')
merged_full[['Name', 'Age', 'Team_from', 'League_to',
             'overall', 'potential', 'Market_value_mln', 'Transfer_fee_mln']].head()

In [ ]:
# Cross-dataset correlation matrix: all model-relevant numeric features
feat_cols = ['Age', 'Market_value_mln', 'Transfer_fee_mln', 'overall', 'potential']
feat_df = merged_full[feat_cols].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(feat_df.corr(method='pearson'), annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=axes[0])
axes[0].set_title('Pearson correlation')

sns.heatmap(feat_df.corr(method='spearman'), annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=axes[1])
axes[1].set_title('Spearman rank correlation')

plt.suptitle('Cross-dataset: transfers × FIFA ratings (2015–2018)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot of merged features — reveals non-linear relationships and outliers
g = sns.pairplot(feat_df, diag_kind='kde', plot_kws={'alpha': 0.4})
g.fig.suptitle('Cross-dataset pairplot (transfers × FIFA ratings)', y=1.02)
plt.show()

## Group-level fairness correlations

For the fairness audit the key question is: does the relationship between player **ability** (`overall`) and **Transfer fee** hold equally across all groups?

We compute the **Spearman rank correlation** between `overall` and `Transfer_fee_mln` within each subgroup:
- **Nationality** — are some nationalities systematically under/over-priced for their ability?
- **Destination league** — do fees track ability more closely for some leagues?
- **Position group** — is the ability–fee link consistent across GKs, defenders, midfielders, forwards?

A group with a low or negative Spearman *r* suggests that ability is a weak or perverse predictor of fee for players in that group — a candidate fairness concern.

In [ ]:
from scipy import stats

# Spearman(overall, Transfer_fee_mln) by nationality — top 15 by transfer count
if 'Nationality' in merged_full.columns:
    nat_counts = merged_full['Nationality'].value_counts()
    top_nats = nat_counts[nat_counts >= 5].index.tolist()[:15]

    nat_corr = []
    for nat in top_nats:
        sub = (merged_full[merged_full['Nationality'] == nat]
               [['overall', 'Transfer_fee_mln']].dropna())
        if len(sub) >= 5:
            r, p = stats.spearmanr(sub['overall'], sub['Transfer_fee_mln'])
            nat_corr.append({'Nationality': nat, 'n': len(sub), 'spearman_r': r, 'p_value': p})

    nat_corr_df = pd.DataFrame(nat_corr).sort_values('spearman_r', ascending=True)

    plt.figure(figsize=(8, 6))
    colors = ['#d73027' if r < 0 else '#4575b4' for r in nat_corr_df['spearman_r']]
    plt.barh(nat_corr_df['Nationality'], nat_corr_df['spearman_r'], color=colors)
    plt.axvline(0, color='black', linewidth=0.8)
    plt.xlabel('Spearman r  (overall vs. Transfer_fee_mln)')
    plt.title('Ability–Fee correlation by Nationality\n(top 15 by transfer count, min 5 records)')
    for i, row in nat_corr_df.reset_index(drop=True).iterrows():
        plt.text(0.01, i, f"n={int(row['n'])}", va='center', fontsize=8)
    plt.tight_layout()
    plt.show()
    display(nat_corr_df.reset_index(drop=True))
else:
    print('Nationality column not available in merged_full.')

In [ ]:
# Spearman(overall, Transfer_fee_mln) by destination league (League_to)
league_counts = merged_full['League_to'].value_counts()
top_leagues = league_counts[league_counts >= 5].index.tolist()[:10]

league_corr = []
for league in top_leagues:
    sub = (merged_full[merged_full['League_to'] == league]
           [['overall', 'Transfer_fee_mln']].dropna())
    if len(sub) >= 5:
        r, p = stats.spearmanr(sub['overall'], sub['Transfer_fee_mln'])
        league_corr.append({'League_to': league, 'n': len(sub), 'spearman_r': r, 'p_value': p})

league_corr_df = pd.DataFrame(league_corr).sort_values('spearman_r', ascending=True)

plt.figure(figsize=(8, 5))
colors = ['#d73027' if r < 0 else '#4575b4' for r in league_corr_df['spearman_r']]
plt.barh(league_corr_df['League_to'], league_corr_df['spearman_r'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Spearman r  (overall vs. Transfer_fee_mln)')
plt.title('Ability–Fee correlation by Destination League (top 10)')
for i, row in league_corr_df.reset_index(drop=True).iterrows():
    plt.text(0.01, i, f"n={int(row['n'])}", va='center', fontsize=8)
plt.tight_layout()
plt.show()
display(league_corr_df.reset_index(drop=True))

In [ ]:
# Spearman(overall, Transfer_fee_mln) by broad position group
pos_map = {
    r'(?i)goalkeeper': 'GK',
    r'(?i)(centre.back|right.back|left.back|defender|wing.back)': 'DF',
    r'(?i)(midfielder|midfield|defensive mid|central mid|attacking mid)': 'MF',
    r'(?i)(forward|winger|striker|centre.forward)': 'FW',
}

def map_position(pos):
    if pd.isna(pos):
        return 'Other'
    for pattern, group in pos_map.items():
        if re.search(pattern, pos):
            return group
    return 'Other'

merged_full['PositionGroup'] = merged_full['Position'].apply(map_position)

pos_corr = []
for pos in merged_full['PositionGroup'].unique():
    sub = (merged_full[merged_full['PositionGroup'] == pos]
           [['overall', 'Transfer_fee_mln']].dropna())
    if len(sub) >= 5:
        r, p = stats.spearmanr(sub['overall'], sub['Transfer_fee_mln'])
        pos_corr.append({'PositionGroup': pos, 'n': len(sub), 'spearman_r': r, 'p_value': p})

pos_corr_df = pd.DataFrame(pos_corr).sort_values('spearman_r', ascending=True)

plt.figure(figsize=(7, 4))
colors = ['#d73027' if r < 0 else '#4575b4' for r in pos_corr_df['spearman_r']]
plt.barh(pos_corr_df['PositionGroup'], pos_corr_df['spearman_r'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Spearman r  (overall vs. Transfer_fee_mln)')
plt.title('Ability–Fee correlation by Position Group')
for i, row in pos_corr_df.reset_index(drop=True).iterrows():
    plt.text(0.01, i, f"n={int(row['n'])}", va='center', fontsize=8)
plt.tight_layout()
plt.show()
display(pos_corr_df.reset_index(drop=True))